---
# Configuration file for post-processing Salish Sea Model results
## Setup for Whidbey Basin WWTP model run scenarios


In [1]:
import yaml
import os
import warnings

import numpy as np

## Define main directory 

In [2]:
# Main dictionary used to output values to SSM_config.yaml
ssm = {}
casename = "ecy21"

## Create array of percentages for depth levels
Multiply these values by the total depth of the water column to get the layer thickness for each model level. This layer thickness is used to calculate volume days.
These values were provided by Su Kyong Yun in her script `volume_calculation.py` on 6/9/2022.

In [3]:
ssm['siglev_diff']=np.array(
    [ 3.2,  5.7,  7.5,  8.9, 10.1, 11.1, 12.1, 13. , 13.8, 14.6]
).tolist() #b/c safedump dosn't allow for objects
# # Updated values given by Su Kyong August 10th, 2022, via Teams chat. 
# ssm['siglev_diff']=numpy.array(
#     [3.2, 5.8, 7.4, 9, 9.8, 11.4, 11.8, 13.4, 13.4, 15]
# ).tolist() #b/c safedump dosn't allow for objects

In [4]:
assert sum(ssm['siglev_diff']) == 100, sum(ssm['siglev_diff'])

## Define location(s) for model output and graphics files

Most output paths are relative now to make things simpler. Just put the generated case file in the directory you want to work from, and call scripts with the file name itself as the "case."

In [5]:
#myhome = '/gscratch/ssmc/USRS/PSI/Ben/'
myhome = '/home/benr/wqmodels/ssm/psi/'
stefano = '/gscratch/ssmc/USRS/PSI/Stefano/'

# Where Stefano keeps the Ecology model outputs
model_outputs = stefano + 'projects/KingCounty/SalishSeaModel/'

# Where the input files are
inputs = myhome + 'ecology_inputs/'

ssm['paths'] = {}
ssm['paths']['model_output'] = {
    # Model output files are listed in the same order in which they appear in the
    # run_tag dictionary in the following cell.
    casename: [
        model_outputs + 'wqm_baseline/ssm_hotstart_wqm_baseline.nc',
        model_outputs + 'wqm_reference/ssm_hotstart_wqm.nc',
    ]
}

ssm['paths']['processed_output'] = 'SSM_data/'
ssm['paths']['spreadsheets'] = 'SSM_output/spreadsheets/'
ssm['paths']['shapefiles'] = 'SSM_output/shapefiles/'
ssm['paths']['graphics'] = 'SSM_output/graphics/'
ssm['paths']['movies'] = 'SSM_output/movies/'

# Define location and name of Shapefile to use for planar graphics
# This version corrects typos in region names and imposes a uniform format for 
# region names
ssm['paths']['shapefile'] = (
    # This file is not right, it has too many rows
    #stefano + 'projects/KingCounty/SalishSeaModel-grid/shapefiles/SSMGrid2_tce_ecy_node_info_v2_10102022/SSMGrid2_tce_ecy_node_info_v2_10102022.shp'
    myhome + 'SalishSeaModel-grid/shapefiles/SSMGrid2_tce_ecy_node_info_v2_10102022/SSMGrid2_tce_ecy_node_info_v2_10102022.shp'
)

# For 303(d) regridding per Figueroa-Kaminsky et al 2025 (not used in this study)
#ssm['paths']['303d'] = {
#    'shapefile': myhome + 'SSMgridExport_303d_20260603/ExistingConditions_2014_update_Ecy2025_clean_20260603.shp',
#    'regrid': myhome + 'SSMgridExport_303d_20260603/SSM_303d_lookup_TCEproportion_Ecy2025_20260501.csv'
#}

# Observations for skill assessment
ssm['paths']['pairings_file'] = myhome + 'projects/Ecy21_analysis/SSM_Opt2_Year_2014_Exist_Model_Obs_Paired.xlsx'

# Nutrient loading input files
ssm['paths']['nutrient_loading_inputs']={
    #'wqm_baseline': inputs + '2021_year1/wqm/cequalicm_wq_2014_Exist3_v3.dat',
    'wqm_baseline': '/home/benr/wqmodels/ssm/ecology_inputs_year1/wqm/cequalicm_wq_2014_Exist3_v3.dat',
    #'wqm_reference': inputs + '2021_year1/wqm/cequalicm_wq_2014_ref3.dat'
    'wqm_reference': '/home/benr/wqmodels/ssm/ecology_inputs_year1/wqm/cequalicm_wq_2014_ref3.dat'
}

## Run information

In [6]:
ssm['run_information'] = {
    # Number of spin-up days removed from model output in post-processing
    'spin_up_days': 5
}

# Run descriptions and names
# These names need to match the "run type" passed to or detected by process_netcdf.py
# Unless you want to change all the keys in this file, either put the model results in
# directories with these names or use the new --run-type option in process_netcdf.py to
# override them

# Special tags for referencing the baseline and reference runs.
ssm['run_information']['baseline'] = 'wqm_baseline'
ssm['run_information']['reference'] = 'wqm_reference'

# Descriptions for all of the runs in the study.
ssm['run_information']['run_description_short'] = {
    casename: {
        'wqm_baseline':'2014 conditions',
        'wqm_reference':'Reference'
    }
}

ssm['run_information']['run_tag'] = {
    casename: {
        'wqm_baseline':'2014 Conditions',
        'wqm_reference':'Reference'
    }
}

Check that required files/output directories exist

In [7]:
for v in ssm['paths']['model_output'].values():
    for f in v:
        assert os.path.isfile(f), f
for d in ('processed_output','graphics','movies','spreadsheets','shapefiles'):
    if not os.path.isdir(ssm['paths'][d]):
        warnings.warn(f"{ssm['paths'][d]} not found, it will be created automatically later")

assert os.path.isfile(ssm['paths']['shapefile']), ssm['paths']['shapefile']
if '303d' in ssm['paths']:
    assert os.path.isfile(ssm['paths']['303d']['shapefile']), ssm['paths']['303d']['shapefile']
    assert os.path.isfile(ssm['paths']['303d']['regrid']), ssm['paths']['303d']['regrid']

if 'pairings_file' in ssm['paths']:
    assert os.path.isfile(ssm['paths']['pairings_file']), ssm['paths']['pairings_file']

if 'nutrient_loading_inputs' in ssm['paths']:
    for f in ssm['paths']['nutrient_loading_inputs'].values():
        assert os.path.isfile(f), f

/tmp/ipykernel_127307/3374632591.py:6: UserWarning: SSM_data/ not found, it will be created automatically later
  warnings.warn(f"{ssm['paths'][d]} not found, it will be created automatically later")
/tmp/ipykernel_127307/3374632591.py:6: UserWarning: SSM_output/graphics/ not found, it will be created automatically later
  warnings.warn(f"{ssm['paths'][d]} not found, it will be created automatically later")
/tmp/ipykernel_127307/3374632591.py:6: UserWarning: SSM_output/movies/ not found, it will be created automatically later
  warnings.warn(f"{ssm['paths'][d]} not found, it will be created automatically later")
/tmp/ipykernel_127307/3374632591.py:6: UserWarning: SSM_output/spreadsheets/ not found, it will be created automatically later
  warnings.warn(f"{ssm['paths'][d]} not found, it will be created automatically later")
/tmp/ipykernel_127307/3374632591.py:6: UserWarning: SSM_output/shapefiles/ not found, it will be created automatically later
  warnings.warn(f"{ssm['paths'][d]} not 

## Save specifications to file
-Use `sort_keys=False` to preserve dictionary order

In [8]:
with open(f'SSM_config_{casename}.yaml', 'w') as file:
    document = yaml.safe_dump(ssm, file,sort_keys=False)